# Integridad · Ciclo, métricas, sesión y textos

[Inicio](../../tests.ipynb) · Anterior: [Integridad · Salas y borrados](salas_y_borrados.ipynb) · Siguiente: [Usabilidad · Aceptación manual](../usabilidad.ipynb)

Las fichas incluyen los criterios del Plan Maestro. Las referencias U/G/F/W/D y las rutas abreviadas en **Evidencia** se interpretan en la [matriz maestra](../../docs/test/qa-master-matrix.md). Una ejecución local no implica un resultado publicado ni prueba un control de entrega.

## RI-09

- **Criterio:** La sala solo pasa de “esperando” a “en curso” y vuelve a “esperando” al terminar las rondas.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: BD: enum + Servidor.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): Inicio, rondas y vuelta a espera en `test_ws_rooms.py` (**U**), que también cubre recuperación por abandono; la recuperación no se presenta como final normal.
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Formato de verificación:** No se identificó Gherkin ejecutable para esta ficha; las comprobaciones se documentan en la evidencia y, cuando aplica, en la reproducción de este cuaderno.

## RI-10

- **Criterio:** El tiempo ideal de un texto lo actualiza solo el servidor cuando alguien lo supera; si no se ha jugado, vale N/5 s (60 PPM).
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: Servidor.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**P**): `test_race_scoring.py`, `test_practice_matches.py` y `test_ws_rooms.py` (**U**) cubren tiempo base de 60 PPM y actualizaciones de récord. `test_rooms_postgresql.py` confirmó en PostgreSQL migrado a `0006` el tiempo y titular persistidos en seis subcasos: tiempo inferior, igual o superior al tiempo base y a la marca almacenada. Falta verificar explícitamente que un valor de récord suministrado por el cliente no modifica el cálculo del servidor.
  
  Evidencia local posterior o adicional: La primera prueba encontró que práctica aceptaba 6.401 ms frente a una base de 6.400 ms (Engram #1001). Corridas posteriores verificaron que práctica (cinco subcasos) y carrera (seis) solo actualizan la marca si el tiempo es estrictamente menor. La API de práctica rechazó `idealTimeMs`, `recordUserId` y `score` con 422 sin escribir; 6.399 ms válidos persistieron con puntaje 60 (Engram #1006 y #1008). [Fuente de prueba](../../backend/tests/integration/test_practice_postgresql.py).
- **Brecha o límite:** Los intentos bloqueados por la preparación del entorno (#1003 y #1005) no cuentan como validaciones.

**Formato de verificación:** No se identificó Gherkin ejecutable para esta ficha; las comprobaciones se documentan en la evidencia y, cuando aplica, en la reproducción de este cuaderno.

## RI-11

- **Criterio:** Las métricas son válidas (tiempo > 0, PPM ≥0, precisión 0–100 %, progreso 0–1) y el puntaje sigue la fórmula del proyecto.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: Servidor (Pydantic).
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**P**): Fórmulas y validaciones representativas en `test_metrics.py`, `test_race_scoring.py`, `test_practice_matches.py` y `test_ws_rooms.py` (**U**). Falta un barrido explícito de todos los límites inclusivos y valores inválidos/no finitos pertinentes.
  
  Evidencia local posterior o adicional: Los límites y valores inválidos se comprobaron en la prueba de esquemas; la corrida mixta tuvo 23/24 pruebas correctas y el único test fallido correspondió a dos subcasos de RI-10, no a RI-11 (Engram #1001). Se complementa con las fórmulas probadas en el corte. [Fuente de prueba](../../backend/tests/test_result_schema.py).
- **Brecha o límite:** La prueba de esquema no mide ni renderiza la interfaz.

**Formato de verificación:** No se identificó Gherkin ejecutable para esta ficha; las comprobaciones se documentan en la evidencia y, cuando aplica, en la reproducción de este cuaderno.

## RI-12

- **Criterio:** El ranking excluye las partidas con precisión menor al 75 %.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: Servidor (consulta).
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): Misma ejecución que TC-FUNC-11 (**U**); el caso funcional y la regla tienen IDs propios, no son dos ejecuciones. [Fuente de prueba](../../backend/tests/test_practice_matches.py). Véase [TC-FUNC-11](../funcional/hu04_ranking.ipynb#TC-FUNC-11); comparten ejecución.
- **Brecha o límite:** Comparte evidencia con TC-FUNC-11; no representa una segunda ejecución.

**Formato de verificación:** No se identificó Gherkin ejecutable para esta ficha; las comprobaciones se documentan en la evidencia y, cuando aplica, en la reproducción de este cuaderno.

## RI-14

- **Criterio:** Un token vencido, adulterado o de un usuario inexistente responde 401 y el frontend cierra la sesión.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: Servidor + Frontend.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**P**): `features/sessions.feature` (**G**) verifica 401 para JWT vencido, adulterado y usuario inexistente. `frontend/e2e/{session,active-expiry}.feature` (**G**) cubre salida por expiración/revocación; falta comprobar en navegador la salida con credencial adulterada y usuario inexistente.
  
  Evidencia local posterior o adicional: Además de la expiración del corte, Chromium visible aprobó 2 escenarios y 12 pasos para firma adulterada y cuenta eliminada: HTTP 401, aviso, redirección a login, credenciales eliminadas y sin renderizado privado (Engram #1018). [Fuente de prueba](../../frontend/e2e/session.feature).
- **Brecha o límite:** Un intento previo se detuvo al leer el cuerpo tras navegar (Engram #1017); no se cuenta como aprobación.

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/sessions.feature).

```gherkin
@auth @sessions @RI-14
Feature: Protected HTTP sessions

  Scenario Outline: Invalid session
    Given a logged-in fixture with a "<credential>" credential
    When it requests a private profile and tries to create a room
    Then both requests return only a bearer 401 and no room is created

    Examples:
      | credential       |
      | absent           |
      | invalid          |
      | tampered         |
      | expired JWT      |
      | idle-expired     |
      | absolute-expired |
      | revoked          |
      | nonexistent user |
      | legacy           |
      | unknown session  |
      | mismatched user  |

  Scenario: Reads do not renew a session
    Given a logged-in fixture with a "valid" credential
    When it reads its profile just before the idle deadline
    Then it is rejected at the original deadline and a fresh login works
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/room_sessions.feature).

```gherkin
@auth @room_sessions @RI-14
Feature: Room sockets follow the server session lifecycle

  Scenario Outline: A room member cannot connect with an ended session
    Given a room with two signed-in members
    When the host session ends by <reason> before connecting
    Then the room handshake closes as unauthorized

    Examples:
      | reason          |
      | logout          |
      | idle expiry     |
      | absolute expiry |

  Scenario: Polls and pings do not save a silent host socket from idle expiry
    Given a room with two signed-in members
    And both members are connected
    When the host polls and sends a ping just before idle expiry
    And the host reaches idle expiry while silent
    Then only the host socket closes as unauthorized
    And the peer can still change readiness

  Scenario: Logging out closes an already connected silent socket
    Given a room with two signed-in members
    And both members are connected
    When the host logs out without sending another frame
    Then only the host socket closes as unauthorized
    And the peer can still change readiness

  Scenario: Only advancing race progress renews within the established cadence
    Given a live race with two signed-in members
    When the host advances near idle expiry
    Then its idle deadline moves by fifteen minutes from receipt
    And further advances respect the one-minute cadence
    And repeated progress does not renew

  Scenario: Advancing race progress cannot extend the absolute deadline
    Given a live race with a sixteen-minute host session
    When the host advances near idle expiry
    Then its renewal is capped by absolute expiry
    And its socket closes at absolute expiry while the peer stays connected
    And the peer can finish the race after the host disconnects
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../backend/tests/features/practice_activity.feature).

```gherkin
@auth @sessions @RI-14
Feature: Practice activity renews a live HTTP session

  Scenario: Confirmed activity renews idle, but polling and repeated notices do not
    Given a logged-in fixture with a "valid" credential
    When a practice notice arrives 14 minutes after login
    Then its confirmed idle deadline is 29 minutes after login
    And a new notice 30 seconds later does not extend it
    And repeating the original notice two minutes later does not extend it
    And polling before that deadline succeeds but activity at it is rejected

  Scenario: A notice near absolute expiry cannot extend beyond it
    Given a logged-in fixture with a "valid" credential
    When practice activity continues until one minute before absolute expiry
    Then the confirmed deadline is capped and a notice at absolute expiry is rejected

  Scenario: Delayed activity can renew only when received before idle expiry
    Given a logged-in fixture with a "valid" credential
    Then a delayed notice received before expiry uses the server receipt time
    And a newer notice at the minute boundary extends idle once
    And content is not accepted in an activity notice

  Scenario Outline: Invalid sessions cannot be renewed
    Given a logged-in fixture with a "<credential>" credential
    Then a practice notice is rejected

    Examples:
      | credential       |
      | absent           |
      | expired JWT      |
      | revoked          |
      | idle-expired     |
      | mismatched user  |
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/session.feature).

```gherkin
@TC-AUTO-01 @RI-14
Feature: Browser session lifecycle
  Background:
    Given I register a new account in the browser

  @TC-FUNC-01
  Scenario: Reload, sign out, go Back and sign in again
    When I reload the profile and the API validates my session
    When I sign out with another tab open and a separate session for my account
    Then only the browser session is revoked
    And the other tab detects the revocation on a protected request and on reload
    Then I can go Back without seeing private data and sign in with a new session

  Scenario: Local sign-out does not wait for revocation
    When my logout request loses connectivity
    Then local sign-out finishes first and warns if revocation cannot be confirmed

  Scenario Outline: Reload validation is temporarily unavailable
    When I reload with a <failure> validation failure
    Then my credentials are retained and the profile stays hidden until retry succeeds

    Examples:
      | failure |
      | network |
      | 503     |

  @invalid_session_reload
  Scenario Outline: Reload rejects an invalid session without exposing private data
    Given my browser session has "<credential>"
    When I reload the profile with the rejected credentials
    Then the API rejects the session and the browser signs out without displaying private data

    Examples:
      | credential                 |
      | an invalid token signature |
      | a deleted account          |
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/active-expiry.feature).

```gherkin
@RI-14 @active_expiry
Feature: Absolute session expiry during active play

  Scenario: Offline practice warns and ends without saving
    Given I register a new account in the browser
    And I start an active practice attempt near absolute expiry
    When I cross the five-minute boundary offline
    Then I see an expiry warning without losing typing focus
    When the browser reaches the absolute deadline
    Then I am signed out with the notice "Tu sesión ya no es válida. Inicia sesión de nuevo. Tu prueba de práctica no se guardó."
    And no practice attempt was submitted
    And signing in again does not resume play

  Scenario Outline: A racing session ends without interrupting its peer
    Given I register a new account in the browser
    And I enter an active race with a separate peer near absolute expiry
    When I cross the five-minute boundary offline
    Then I see an expiry warning without losing typing focus
    When <ending>
    Then I am signed out with the notice "Your session expired. Sign in again to rejoin the room."
    And the peer can finish the race
    And signing in again does not resume play

    Examples:
      | ending                                    |
      | the browser reaches the absolute deadline |
      | the server revokes my racing session      |
```

**Gherkin ejecutable (original en inglés)** · [Fuente](../../frontend/e2e/practice-activity.feature).

```gherkin
@RI-14 @practice-activity
Feature: Practice typing sends minimal session activity

  Scenario: Only real typing sends a notice, at most once a minute
    Given I register a new account in the browser
    When I start a practice test with a controlled clock
    Then advancing the clock without typing sends nothing
    And real typing sends a content-free notice and respects the minute boundary
```

Se incluyen las funcionalidades etiquetadas con `@RI-14`. Los escenarios de lectura, renovación, WebSocket y fallas transitorias también son regresiones de sesión; no equivalen al criterio de 401 y cierre de la interfaz.

## RI-15

- **Criterio:** Un texto solo se guarda si pasa la validación, y su dificultad se calcula siempre con las mismas reglas de clasificación.
- **Método previsto:** API, WebSocket o SQL, según la capa. Garantía prevista: Servidor + BD.
- **Estado:** **V — Verificado**.
- **Evidencia:** En la matriz (**V**): Validación, clasificación/calibración, deduplicación, persistencia y rollback: `services/texts_handler/test_{validation_normalization,classification,persistence}.py` e `integration/test_text_postgresql.py` (**U**). No requiere Gemini real. [Fuente de prueba](../../backend/tests/integration/test_text_postgresql.py).
- **Brecha o límite:** No se identificó una brecha para el escenario y entorno documentados.

**Formato de verificación:** No se identificó Gherkin ejecutable para esta ficha; las comprobaciones se documentan en la evidencia y, cuando aplica, en la reproducción de este cuaderno.

## Reproducción local opcional

Prepare el entorno desechable de la [guía de pruebas](../../test/README.md). Ejecute solo la celda que necesita, con el kernel iniciado en la raíz de Typr o dentro de `notebooks/`. No ejecute en paralelo. Abrir el cuaderno no inicia pruebas; la salida describe una corrida local, no cambia el estado de las fichas. Chromium se abre en una sesión gráfica local. Las celdas de backend ejecutan la suite indicada completa, no solo los casos de esta página.

### Criterios de aceptación · RI-09 a RI-12 y RI-15

**Plan (§5.6):** La sala pasa de espera a carrera y vuelve a espera al finalizar las rondas; el servidor calcula el tiempo base (N/5 s) y solo lo actualiza al superarlo. El tiempo debe ser positivo, PPM no negativo, precisión de 0 a 100 % y progreso de 0 a 1; el puntaje sigue la fórmula del proyecto. El ranking excluye precisiones inferiores al 75 %; solo se guardan textos válidos, con dificultad clasificada de forma consistente.

**Qué comprueba esta celda:** Ejecuta toda la suite de pruebas unitarias, integración y migraciones del backend. Las aserciones pertinentes comprueban el retorno a espera y la limpieza del estado de listo, el tiempo base y la mejora estricta del récord, los límites del esquema y el rechazo de métricas no finitas, el puntaje en casos de práctica y carrera, la inclusión del 75 % exacto y la persistencia de textos válidos con la dificultad esperada.

**Límite:** Los casos de esquema y puntaje no miden ni renderizan la interfaz; RI-14 se reproduce en la siguiente celda.

In [ ]:
from pathlib import Path
from runpy import run_path

directorio = Path.cwd().resolve()
for raiz in (directorio, *directorio.parents):
    if (raiz / "docker-compose.test.yml").is_file() and (raiz / "notebooks/ejecutar_qa.py").is_file():
        break
else:
    raise RuntimeError("Inicie el kernel desde la raíz de Typr o dentro de notebooks/.")

run_path(str(raiz / "notebooks/ejecutar_qa.py"))["ejecutar"]('backend-unit-integration-migrations')

### Criterios de aceptación · RI-14

**Plan (§5.6):** Ante un token vencido, adulterado o de un usuario inexistente, la API responde 401 y la interfaz cierra la sesión.

**Qué comprueba esta celda:** El filtro `@RI-14` selecciona 11 escenarios de `session.feature`, `active-expiry.feature` y `practice-activity.feature`. La recarga con firma inválida o cuenta borrada comprueba 401 con `WWW-Authenticate: Bearer`, eliminación de credenciales y ausencia de contenido privado; otros escenarios comprueban expiración durante el juego, actividad, rutas, cierre de sesión y fallos transitorios al validar.

**Límite:** Los dos ejemplos de `@invalid_session_reload` no incluyen un token vencido. La inactividad vencida con 401 se comprueba por separado en `@profile_routes`; la expiración absoluta durante el juego se comprueba en `@active_expiry`, no como la misma recarga HTTP.

In [ ]:
from pathlib import Path
from runpy import run_path

directorio = Path.cwd().resolve()
for raiz in (directorio, *directorio.parents):
    if (raiz / "docker-compose.test.yml").is_file() and (raiz / "notebooks/ejecutar_qa.py").is_file():
        break
else:
    raise RuntimeError("Inicie el kernel desde la raíz de Typr o dentro de notebooks/.")

run_path(str(raiz / "notebooks/ejecutar_qa.py"))["ejecutar"]('browser-gherkin', etiquetas='@RI-14')

[Inicio](../../tests.ipynb) · Anterior: [Integridad · Salas y borrados](salas_y_borrados.ipynb) · Siguiente: [Usabilidad · Aceptación manual](../usabilidad.ipynb)